# RR ETDs to MARC

This script retrieves OAI metadata for the ETDs located in WVU's Research Repository and uses an XSLT script to transform the metadata into MARC/XML. It can then perform an API call to validate each record and upload the valid records into WMS.

DO NOT run this script in entirety unless you want to automatically upload records to OCLC. It is recommended to run each cell individually.

Outputs:
- XML:
  - OAI metadata that is input to the XSLT script
  - Complete MARC output file
  - Invalid MARC records
  - Valid MARC records

- XLSX:
  - Embargoed records that were not processed or uploaded
  - List of invalid records that were not uploaded
  - List of records missing DOIs that were not uploaded


## Dependencies

In [ ]:
# connect to Google Drive. The XSLT stylesheet must be in your Google Drive to run this script.
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install saxonche
from saxonche import *

from lxml import etree

from datetime import datetime

!pip install bookops-worldcat
from bookops_worldcat import MetadataSession, WorldcatAccessToken

from io import BytesIO, StringIO

!pip install sickle
from sickle import Sickle
from sickle.iterator import OAIResponseIterator

import requests
import yaml
import csv
import json
import pandas as pd

!pip install xlsxwriter
from xlsxwriter import Workbook

from logging import exception

# Retrieve OAI metadata

## !! ACTION REQUIRED: update from and until dates and file names and locations

In [ ]:
# OAI call to the RR. Update dates for the set of ETDs you want to upload
sickle = Sickle('https://researchrepository.wvu.edu/do/oai', iterator=OAIResponseIterator)

responses = sickle.ListRecords(**{'metadataPrefix': 'document-export', 'set':'publication:etd',
                                  'from':'2025-08-28', 'until':'2025-09-02'})
# API config file
configfile = "/content/drive/My Drive/ETD2MARC/metadata_api_config.yml"

# location of XSLT file
XSLTfile = "/content/drive/My Drive/ETD2MARC/RRETD2MARC_2025_12_08.xsl"

# OAI metadata that is run through the XSLT script (does not include records with no DOI or with a future embargo date)
saveFile = "/content/drive/My Drive/ETD2MARC/2025_Summer/2025_Summer_ETDs.xml"
inputXMLfile = saveFile

# direct output of XSLT script - all MARC/XML records
outputXMLfile = "/content/drive/My Drive/ETD2MARC/2025_Summer/2025_Summer_MARC.xml"

# only valid MARC/XML records
validXMLfile = "/content/drive/My Drive/ETD2MARC/2025_Summer/2025_Summer_valid_MARC.xml"
# only invalid MARC/XML records
invalidXMLfile = "/content/drive/My Drive/ETD2MARC/2025_Summer/2025_Summer_invalid_MARC.xml"

# list of records missing DOIs
missingfile = "/content/drive/My Drive/ETD2MARC/2025_Summer/2025_Summer_missingDOIs_list.xlsx"
# list of invalid MARC records
invalidfile = "/content/drive/My Drive/ETD2MARC/2025_Summer/2025_Summer_invalid_list.xlsx"
# list of records successfully uploaded
uploadedfile = "/content/drive/My Drive/ETD2MARC/2025_Summer/2025_Summer_uploaded_list.xlsx"
# list of records with active embargo dates that were not processed
embargofile = "/content/drive/My Drive/ETD2MARC/2025_Summer/2025_Summer_embargo_list.xlsx"
# list of records that failed to upload to WMS
errorfile = "/content/drive/My Drive/ETD2MARC/2025_Summer/2025_Summer_error_list.xlsx"
# list of records successfully uploaded to WMS
processedfile = "/content/drive/My Drive/ETD2MARC/2025_Summer/2025_Summer_processed_list.xlsx"

Save XML as etree

In [ ]:
NSMAP = {None : 'http://www.openarchives.org/OAI/2.0/',
        'doc' : ''}

root = etree.Element("{http://www.openarchives.org/OAI/2.0/}OAI-PMH", nsmap=NSMAP)
records = etree.Element("{http://www.openarchives.org/OAI/2.0/}ListRecords", nsmap=NSMAP)


missinglist = []
embargolist = {
    'doi':[],
    'embargo_date':[]
}

for response in responses:

    xml = response.xml

    for ListRecords in xml:

      for record in ListRecords:
        # check that DOI exists
        if record.xpath(".//*[local-name() = 'field'][@name='doi']"):
          # check if embargo date exists
          if record.xpath(".//*[local-name() = 'field'][@name='embargo_date']"):
              embargo_date = record.xpath(".//*[local-name() = 'field'][@name='embargo_date']/*[local-name() = 'value']/text()")[0][:10]
              # do not process if embargo date is in the future
              if datetime.strptime(embargo_date, '%Y-%m-%d').date() > datetime.today().date():
                embargolist['doi'].append(record.xpath(".//*[local-name() = 'field'][@name='doi']/*[local-name() = 'value']/text()")[0])
                embargolist['embargo_date'].append(embargo_date)
              else:
                # otherwise add record to lxml etree to be run through the XSLT
                records.append(record)
          else:
            # otherwise add record to lxml etree to be run through the XSLT
              records.append(record)
        # do not process records with no DOI - add to list
        elif record.xpath(".//*[local-name() = 'coverpage-url']"):
          missinglist.append(record.xpath(".//*[local-name() = 'coverpage-url']/text()")[0])

# write list of missing DOIs and list of embargos to excel
df = pd.DataFrame(missinglist)
df.to_excel(missingfile, index=False)

df2 = pd.DataFrame(embargolist)
df2.to_excel(embargofile, index=False)

# add records with DOIs and no or past embargo dates to be processed
root.append(records)
etree.cleanup_namespaces(root)

# save records to be processed as XML file
with open(saveFile, 'wb') as fp:
  fp.write(etree.tostring(root, pretty_print="true", encoding="utf-8"))

## Run XSLT script to produce MARC

In [ ]:
# run XSLT script
proc = PySaxonProcessor(license=False)

xsltproc = proc.new_xslt30_processor()
document = proc.parse_xml(xml_file_name=inputXMLfile)
executable = xsltproc.compile_stylesheet(stylesheet_file=XSLTfile)

output = executable.transform_to_string(xdm_node=document)

# save MARC/XML output to file
with open(outputXMLfile, "w") as file:
    file.write(output)

## API call

In [ ]:
# set up lxml etrees for valid and invalid MARC
marcNSMAP = {'marc' : 'http://www.loc.gov/MARC21/slim'}
invalidRoot = etree.Element("{http://www.loc.gov/MARC21/slim}collection", nsmap=marcNSMAP)
validRoot = etree.Element("{http://www.loc.gov/MARC21/slim}collection", nsmap=marcNSMAP)
errorRoot = etree.Element("{http://www.loc.gov/MARC21/slim}collection", nsmap=marcNSMAP)

In [ ]:
# start API session
with open(configfile, 'r') as stream:
    config = yaml.safe_load(stream)

    token = WorldcatAccessToken(
        key= config.get('key'),
        secret= config.get('secret'),
        scopes="WorldCatMetadataAPI:manage_bibs",
    )
    print(token)
    print(token.is_expired())

access_token: 'tk_33DgdaYxJw4fmkF7QWJ44ePyAhhYbdvvdZn9', expires_at: '2025-12-08 14:55:39Z'
False


## VALIDATE MARC RECORD

In [ ]:
# function to retrieve JSON error output from BookOps-WorldCat
# solution from https://www.reddit.com/r/learnpython/comments/s20v6w/easy_way_to_extract_json_part_of_a_string/

def parse_json(s):
    s = s[next(idx for idx, c in enumerate(s) if c in "{["):]
    try:
        return json.loads(s)
    except json.JSONDecodeError as e:
        return json.loads(s[:e.pos])

In [ ]:
invalidList = {
    'url':[],
    'error':[]
}

# validate record
with open(outputXMLfile,"rb") as xml_file:
    session = MetadataSession(authorization=token)
    marcCollection = BytesIO(xml_file.read())
    tree = etree.parse(marcCollection)
    root = tree.getroot()
    for marcRecord in root.iterfind("{http://www.loc.gov/MARC21/slim}record"):
      # if record validates, add to the valid etree
      try:
        response = session.bib_validate(
        record = etree.tostring(marcRecord),
        recordFormat="application/marcxml+xml",
        validationLevel="validateFull",
        )
        print(response.json())
        if response.json()["status"]["summary"] == 'VALID':
          validRoot.append(marcRecord)
      # otherwise add to the invalid etree and add the id and error to the list of invalid records
      except Exception as e:
        error_json = parse_json(str(e))
        print(error_json)

        invalidRoot.append(marcRecord)
        invalidList['url'].append(marcRecord.xpath("./*[local-name() ='datafield'][@tag='856']/*[local-name() = 'subfield'][@code='u']/text()"))
        invalidList['error'].append(error_json['validationErrors']['errors'])

# save invalid list to file
df = pd.DataFrame(invalidList)
df.to_excel(invalidfile, index=False)

# save MARC/XML to files
with open(invalidXMLfile, 'wb') as fp:
  fp.write(etree.tostring(invalidRoot, pretty_print="true", encoding="utf-8"))

with open(validXMLfile, 'wb') as fp:
  fp.write(etree.tostring(validRoot, pretty_print="true", encoding="utf-8"))


{'httpStatus': 'OK', 'status': {'summary': 'VALID', 'description': 'The provided Bib is valid'}}
{'httpStatus': 'OK', 'status': {'summary': 'VALID', 'description': 'The provided Bib is valid'}}
{'httpStatus': 'OK', 'status': {'summary': 'VALID', 'description': 'The provided Bib is valid'}}
{'httpStatus': 'OK', 'status': {'summary': 'VALID', 'description': 'The provided Bib is valid'}}
{'httpStatus': 'OK', 'status': {'summary': 'VALID', 'description': 'The provided Bib is valid'}}
{'httpStatus': 'OK', 'status': {'summary': 'VALID', 'description': 'The provided Bib is valid'}}
{'httpStatus': 'OK', 'status': {'summary': 'VALID', 'description': 'The provided Bib is valid'}}
{'httpStatus': 'OK', 'status': {'summary': 'VALID', 'description': 'The provided Bib is valid'}}
{'httpStatus': 'OK', 'status': {'summary': 'VALID', 'description': 'The provided Bib is valid'}}
{'httpStatus': 'OK', 'status': {'summary': 'VALID', 'description': 'The provided Bib is valid'}}
{'httpStatus': 'OK', 'status':

## !!CAUTION - CREATE MARC RECORD

In [ ]:
errorList = {
    'url':[],
    'error':[]
}

processedList = {
    'OCN':[]
}

session = MetadataSession(authorization=token)

for marcRecord in validRoot.iterfind("{http://www.loc.gov/MARC21/slim}record"):
  try:
    createResponse = session.bib_create(
        record = etree.tostring(marcRecord),
        recordFormat="application/marcxml+xml"
            )
    print(createResponse.content)
    processedList.append()

  except Exception as e:
    error_json = parse_json(str(e))
    print(error_json)

    errorRoot.append(marcRecord)
    errorList['url'].append(marcRecord.xpath("./*[local-name() ='datafield'][@tag='856']/*[local-name() = 'subfield'][@code='u']/text()"))
    errorList['error'].append(error_json)

df = pd.DataFrame(errorList)
df.to_excel(errorfile, index=False)


b'<marc:record xmlns:marc="http://www.loc.gov/MARC21/slim">\n      <marc:leader>00000nam a2200000 a 4500</marc:leader>\n      <marc:controlfield tag="008">251010s2024    wvu     o     000 u und d</marc:controlfield>\n      <marc:controlfield tag="006">m     o  d        </marc:controlfield>\n      <marc:controlfield tag="007">cr n||||||||||</marc:controlfield>\n      <marc:datafield tag="040" ind1=" " ind2=" ">\n         <marc:subfield code="a">WVU</marc:subfield>\n         <marc:subfield code="b">eng</marc:subfield>\n         <marc:subfield code="c">WVU</marc:subfield>\n      </marc:datafield>\n      <marc:datafield tag="100" ind1="1" ind2=" ">\n         <marc:subfield code="a">Bajo, Ka&#699;ala K.,</marc:subfield>\n         <marc:subfield code="e">author.</marc:subfield>\n         <marc:subfield code="1">https://orcid.org/0009-0005-8833-3014</marc:subfield>\n      </marc:datafield>\n      <marc:datafield tag="245" ind1="1" ind2="0">\n         <marc:subfield code="a">Making Something O